# Penyelarasan Visual dalam Kerumunan

**ID proyek:** `O005-LEGA-V101-PRJ06`  
**Status:** titik awal pedagogis yang ditulis secara independen.

Notebook ini menggunakan data sintetis/terbuka saja. Notebook ini **bukan** kode atau data dari makalah yang dikutip dalam bab sumber dan **bukan** klaim reproduksi hasil penelitian mana pun.


## Pertanyaan pemodelan

Seberapa cepat aturan penyelarasan visual lokal menghasilkan gerak kolektif dari arah awal acak?

Tujuan kerja: tetapkan sistem, jalankan eksperimen deterministik, periksa invarian, visualisasikan perilaku, lalu kritik kecukupan model.


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

SEED = 2026082206
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=6, suppress=True)


## Struktur dan asumsi

Agen bergerak pada domain periodik, menggabungkan arah sendiri dengan arah rata-rata tetangga dalam radius visual tetap, dan menerima gangguan sudut kecil.

Semua skala dan parameter di notebook ini bersifat ilustratif. Ubah satu asumsi pada satu waktu dan catat dampaknya pada keluaran serta invarian.


In [ ]:
n_agents, n_steps = 70, 150
positions = rng.uniform(0.0, 1.0, (n_agents, 2))
angles = rng.uniform(-np.pi, np.pi, n_agents)
initial_positions = positions.copy()
initial_angles = angles.copy()
polarization = []
neighbor_counts = []
visual_radius = 0.27

for _ in range(n_steps):
    displacement = positions[None, :, :] - positions[:, None, :]
    displacement = (displacement + 0.5) % 1.0 - 0.5
    neighbors = np.sum(displacement**2, axis=2) <= visual_radius**2
    counts = neighbors.sum(axis=1)
    local_cos = neighbors @ np.cos(angles) / counts
    local_sin = neighbors @ np.sin(angles) / counts
    local_angle = np.arctan2(local_sin, local_cos)
    own = np.column_stack([np.cos(angles), np.sin(angles)])
    target = np.column_stack([np.cos(local_angle), np.sin(local_angle)])
    blended = 0.55 * own + 0.45 * target
    angles = np.arctan2(blended[:, 1], blended[:, 0]) + rng.normal(0.0, 0.025, n_agents)
    positions = (positions + 0.012 * np.column_stack([np.cos(angles), np.sin(angles)])) % 1.0
    polarization.append(float(np.hypot(np.cos(angles).mean(), np.sin(angles).mean())))
    neighbor_counts.append(float(counts.mean()))


## Pemeriksaan numerik

Pemeriksaan berikut sengaja berada di dalam notebook: eksekusi berhenti bila suatu invarian dasar gagal. Ini bukan bukti bahwa model benar; ini hanya bukti bahwa implementasi memenuhi kontrak numerik terbatasnya.


In [ ]:
initial_pol = float(np.hypot(np.cos(initial_angles).mean(), np.sin(initial_angles).mean()))
assert np.all((positions >= 0.0) & (positions < 1.0))
assert polarization[-1] > initial_pol + 0.35
assert np.all((np.asarray(polarization) >= 0.0) & (np.asarray(polarization) <= 1.0 + 1e-12))
assert 1.0 < np.mean(neighbor_counts) < n_agents


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
axes[0].quiver(initial_positions[:, 0], initial_positions[:, 1], np.cos(initial_angles), np.sin(initial_angles), angles="xy", scale=18)
axes[0].set(title="awal", xlim=(0, 1), ylim=(0, 1), aspect="equal")
axes[1].quiver(positions[:, 0], positions[:, 1], np.cos(angles), np.sin(angles), angles="xy", scale=18)
axes[1].set(title="akhir", xlim=(0, 1), ylim=(0, 1), aspect="equal")
axes[2].plot(polarization)
axes[2].set(xlabel="langkah", ylabel="polarisasi", title="keteraturan global", ylim=(0, 1.05))
fig.tight_layout()
plt.show()
plt.close(fig)


## Validasi, identifikasi, dan keterbatasan

Keterbatasan awal: Tetangga ditentukan hanya oleh jarak periodik; tidak ada rintangan, bidang pandang berarah, oklusi, tabrakan, perbedaan kecepatan, atau kepanikan.

Jawab sebelum menafsirkan gambar:

1. Besaran apa yang benar-benar dapat diamati, dan bagaimana galat pengukurannya dimodelkan?
2. Parameter mana yang dapat diidentifikasi dari keluaran tersebut? Tunjukkan dengan profil galat, pemisahan latih/uji, atau eksperimen sensitivitas.
3. Invarian atau pola kualitatif apa yang harus tetap benar ketika ukuran langkah, benih acak, atau resolusi diubah?
4. Temukan satu skenario kegagalan model dan jelaskan data tambahan yang diperlukan untuk membedakannya dari model alternatif.


## Daftar periksa reproduksibilitas

- [ ] Gunakan CPython dan versi paket tepat seperti `requirements.lock`.
- [ ] Jalankan ulang dari kernel kosong tanpa jaringan.
- [ ] Pertahankan nilai `SEED` (benih acak), lalu ulangi dengan sedikitnya lima benih acak lain dan laporkan variasinya.
- [ ] Catat setiap perubahan parameter, persamaan, toleransi, serta pembagian data.
- [ ] Pastikan semua uji lulus dan jelaskan mengapa tiap uji relevan.
- [ ] Simpan hasil turunan di luar notebook sumber; notebook distribusi harus tetap tanpa keluaran tersimpan.
- [ ] Bedakan hasil simulasi, data sintetis, dan klaim empiris secara eksplisit.
